# # 使用贝叶斯优化得到的最佳超参数进行模型训练
#
# 本notebook旨在利用之前贝叶斯优化得到的最佳超参数组合，重新进行一次完整的模型训练、验证和测试。
# 我们将尽可能复用项目中的现有代码模块。


In [ ]:
# 单元格1: 导入模块
import os
import sys
import json
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR, ReduceLROnPlateau
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix, balanced_accuracy_score
import random

# 添加PyTorch序列化安全设置，避免加载模型时报错
try:
    safe_globals = [
        np.dtype,
        np.core.multiarray.scalar,
        np.ndarray,
        np.generic,
        np.float64,
        np.float32,
        np.int64,
        np.int32
    ]
    torch.serialization.add_safe_globals(safe_globals)
    print("✅ 已添加numpy类型到PyTorch安全全局变量列表")
except Exception as e:
    print(f"⚠️ 添加安全全局变量时出错 (可忽略): {e}")

# 导入项目中的模块
from config import load_config, save_config, is_mat_format
from data import load_data
from data.mat_loader import validate_filtered_data
from models import get_model
from utils.metrics import calculate_class_weights, evaluate_model
from utils.visualization import visualize_training_curves, visualize_confusion_matrix
from utils.model_io import save_model_with_architecture


In [ ]:
# 单元格2: 配置设置（修改后）
cfg = load_config()

# 🔧 修改：默认使用MAT格式和固定测试集
# 设置 MAT 文件路径
mat_file_path = "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38.mat"
if not os.path.exists(mat_file_path):
    print(f"错误: MAT文件不存在: {mat_file_path}")
    print("请修改为正确的路径")
    raise FileNotFoundError(f"MAT文件未找到: {mat_file_path}")

cfg['mat_file_path'] = mat_file_path

# 🔧 新增：配置固定测试集（这是新的推荐方式）
cfg['test_prob_idx'] = [13, 23, 38]  # 固定测试集的prob_idx
cfg['train_val_ratio'] = 0.75        # 训练集占训练+验证集的75%

# 可选：使用预设配置
# from config import apply_preset_config
# apply_preset_config(cfg, 'default_fixed_test')  # 使用默认配置
# apply_preset_config(cfg, 'small_test_set')      # 或使用小测试集配置


# 🔧 新增：背景处理配置
cfg['filter_background'] = True  # 或 False，根据需要
cfg['include_background_in_classes'] = False  # 背景是否作为分类类别
cfg['background_label_target'] = -1  # 背景标签目标值

# 🔧 可选：根据背景处理调整类别数
if not cfg.get('filter_background', True) and cfg.get('include_background_in_classes', False):
    cfg['num_class'] = 103  # 包含背景类别
else:
    cfg['num_class'] = 102  # 标准102类别
    
# 清空原版数据目录路径，确保使用MAT格式
cfg['data_dirs'] = {
    'train_dir': None,
    'test_dir': None,
    'val_dir': None
}

# 🔧 移除旧的患者分割配置（已被test_prob_idx替代）
# cfg['dataset_split'] 保留但不再使用

# 设置实验名称
cfg['experiment_name'] = f"BestHyperparams_MAT_FixedTest_{time.strftime('%Y%m%d_%H%M%S')}"

# 确保关键配置
cfg['use_old_zipfile_serialization'] = True
cfg['norm'] = True      # 确保标准化开启
cfg['apply_pca'] = False  # 确保PCA设置正确
cfg['n_pca'] = 0

# 🔧 新增：验证配置
from config import validate_config, print_data_split_info
if validate_config(cfg):
    print_data_split_info(cfg)
else:
    print("❌ 配置验证失败，请检查配置")
    raise ValueError("配置验证失败")

print(f"✅ 配置完成:")
print(f"  MAT文件路径: {cfg.get('mat_file_path')}")
print(f"  数据格式检测: {'MAT格式' if is_mat_format(cfg) else '原版格式'}")
print(f"  测试集划分: {'固定prob_idx' if cfg.get('test_prob_idx') else '随机划分'}")
print(f"  实验名称: {cfg['experiment_name']}")

In [ ]:
# 单元格3: 最佳超参数设置

# 测试不同背景处理模式
background_modes = {
    'filter': {
        'filter_background': True,
        'description': '过滤背景模式（原默认）'
    },
    'ignore': {
        'filter_background': False,
        'include_background_in_classes': False,
        'background_label_target': -1,
        'description': '保留背景但忽略模式'
    },
    'classify': {
        'filter_background': False,
        'include_background_in_classes': True,
        'background_label_target': 0,
        'num_class': 103,
        'description': '背景作为分类类别模式'
    }
}

# 选择模式
mode = 'classify'  # 或 'ignore', 'classify'
cfg.update(background_modes[mode])
print(f"🎯 背景处理模式: {background_modes[mode]['description']}")


best_hyperparams_from_bayesian = {
    'learning_rate': 9.191261837889327e-05,
    'weight_decay': 0.0005175833650131985,
    'optimizer': 'adamw',
    'dropout_rate': 0.19317216698770392,
    'activation': 'gelu',
    'lr_scheduler': 'step',
    'model_type': 'base_mlp',
    'step_size': 4,
    'step_gamma': 0.1590396939768399,
    'layer_sizes_idx': 0,  # 对应 [4096, 4096, 4096, 4096]
}

print("🔧 应用贝叶斯优化的最佳超参数...")

# 更新配置 - 按照main.py的参数映射方式
for key, value in best_hyperparams_from_bayesian.items():
    if key == 'learning_rate':
        cfg['lr'] = value
    elif key == 'lr_scheduler':
        cfg['lr_scheduler_type'] = value
        cfg['use_lr_scheduler'] = True
    elif key == 'step_size':
        # 注意：需要保存为lr_milestones的格式，但StepLR使用step_size
        cfg['lr_step_size'] = value  # 为StepLR准备
        cfg['lr_milestones'] = [value, value*2]  # 为MultiStepLR准备
    elif key == 'step_gamma':
        cfg['lr_gamma'] = value
    elif key == 'layer_sizes_idx':
        # 根据索引设置隐藏层配置 - 与optimization.py中的定义一致
        layer_sizes_options = [
            [4096, 4096, 4096, 4096],  # idx=0
            [3072, 3072, 3072, 3072],  # idx=1  
            [2048, 2048, 2048, 2048],  # idx=2
        ]
        if value < len(layer_sizes_options):
            cfg['hidden_units'] = layer_sizes_options[value]
        else:
            cfg['hidden_units'] = layer_sizes_options[0]  # 默认使用第一个
    else:
        cfg[key] = value

# 确保其他必要参数与main.py一致
cfg['epochs'] = cfg.get('epochs', 30)
cfg['batch_size'] = cfg.get('batch_size', 128)
cfg['val_epochs'] = cfg.get('val_epochs', 3)
cfg['save_checkpoints'] = True
cfg['model_name'] = cfg.get('model_name', 'BrainVoxel_102Class_MLP')
cfg['dataset_name'] = cfg.get('dataset_name', 'BrainVoxel')

print("✅ 最佳超参数配置完成:")
print(f"  模型类型: {cfg['model_type']}")
print(f"  隐藏层: {cfg['hidden_units']}")
print(f"  学习率: {cfg['lr']:.2e}")
print(f"  权重衰减: {cfg['weight_decay']:.2e}")
print(f"  优化器: {cfg['optimizer']}")
print(f"  激活函数: {cfg['activation']}")
print(f"  Dropout率: {cfg['dropout_rate']:.4f}")
print(f"  学习率调度器: {cfg['lr_scheduler_type']}")
if cfg['lr_scheduler_type'] == 'step':
    print(f"    Step Size: {cfg.get('lr_step_size', 4)}")
    print(f"    Gamma: {cfg.get('lr_gamma', 0.1)}")

In [ ]:
# 单元格4: 环境设置（与main.py保持一致）

def setup_environment(config):
    """设置环境，包括随机种子和设备（与main.py一致）"""
    # 设置随机种子
    random.seed(config['random_seed'])
    torch.manual_seed(config['random_seed'])
    torch.cuda.manual_seed(config['random_seed'])
    torch.cuda.manual_seed_all(config['random_seed'])
    np.random.seed(config['random_seed'])
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # 设置设备
    device = torch.device(f"cuda:{config['device']}" if config['device'] >= 0 and torch.cuda.is_available() else "cpu")
    
    print(f"🎯 随机种子设置为: {config['random_seed']}")
    print(f"🖥️ 使用设备: {device}")
    
    return device

device = setup_environment(cfg)

# 创建保存目录
save_dir = os.path.join(cfg.get('save_dir', './results'), cfg['experiment_name'])
cfg['save_dir'] = save_dir
os.makedirs(save_dir, exist_ok=True)
print(f"📁 结果保存目录: {save_dir}")


In [ ]:

# 单元格5: 加载数据（修改后）


# 验证背景处理配置
from config import validate_config, print_background_config_info

if not validate_config(cfg):
    raise ValueError("❌ 配置验证失败，请检查背景处理配置")

print_background_config_info(cfg)

print(f"🔄 开始使用MAT格式加载数据...")
print(f"  MAT文件: {cfg['mat_file_path']}")
print(f"  测试集配置: {cfg.get('test_prob_idx', '随机划分')}")

# 确保MAT文件存在
if not os.path.exists(cfg['mat_file_path']):
    raise FileNotFoundError(f"MAT文件不存在: {cfg['mat_file_path']}")

# 🔧 修改：使用统一接口加载数据（自动检测是否使用固定测试集）
dataset_dict, train_loader, val_loader, test_loader = load_data(cfg, mode='train')

# 🔍 数据完整性验证
print("\n🔍 验证数据完整性:")
validate_filtered_data(dataset_dict['train_samples'], dataset_dict['train_labels'], "训练数据")
validate_filtered_data(dataset_dict['val_samples'], dataset_dict['val_labels'], "验证数据")
validate_filtered_data(dataset_dict['test_samples'], dataset_dict['test_labels'], "测试数据")

# 更新配置中的特征维度
cfg['feature_dim'] = dataset_dict['feature_dim']
cfg['num_class'] = dataset_dict['num_classes']

# 🔍 验证StandardScaler标准化状态
print("\n📊 验证StandardScaler标准化:")
scaler = dataset_dict.get('scaler', None)
if scaler is not None:
    print("✅ 检测到StandardScaler对象:")
    if hasattr(scaler, 'mean_') and hasattr(scaler, 'scale_'):
        print(f"  ✓ Scaler均值范围: [{scaler.mean_.min():.4f}, {scaler.mean_.max():.4f}]")
        print(f"  ✓ Scaler缩放范围: [{scaler.scale_.min():.4f}, {scaler.scale_.max():.4f}]")
    
    # 验证实际标准化效果
    train_mean = np.mean(dataset_dict['train_samples'], axis=0)
    train_std = np.std(dataset_dict['train_samples'], axis=0)
    print(f"  ✓ 实际数据均值范围: [{train_mean.min():.4f}, {train_mean.max():.4f}]")
    print(f"  ✓ 实际数据标准差范围: [{train_std.min():.4f}, {train_std.max():.4f}]")
    
    if abs(train_mean.mean()) < 0.1 and abs(train_std.mean() - 1.0) < 0.2:
        print("  ✅ 数据已正确通过StandardScaler标准化（均值≈0，标准差≈1）")
    else:
        print("  ⚠️ 标准化可能不完整")
else:
    print("⚠️ 未找到StandardScaler对象")

# 🔍 验证数据划分结果（新增）
print(f"\n🎯 验证数据划分结果:")
if cfg.get('test_prob_idx') is not None:
    print(f"  使用固定测试集模式:")
    print(f"  配置的测试集prob_idx: {dataset_dict.get('test_prob_idx_used', 'N/A')}")
    print(f"  实际测试集prob_idx: {dataset_dict.get('test_prob_idx_actual', 'N/A')}")
    print(f"  训练+验证集prob_idx范围: {dataset_dict.get('train_val_prob_idx', 'N/A')}")
    
    # 验证数据集无交叉
    test_actual = set(dataset_dict.get('test_prob_idx_actual', []))
    train_val = set(dataset_dict.get('train_val_prob_idx', []))
    intersection = test_actual.intersection(train_val)
    
    if intersection:
        print(f"  ❌ 警告: 检测到数据集交叉! 交叉的prob_idx: {intersection}")
    else:
        print(f"  ✅ 数据集无交叉验证通过")
else:
    print(f"  使用随机划分模式（原有方式）")

# 🔍 关键验证：检查标签格式和DataLoader输出
print(f"\n🔍 验证DataLoader中的标签格式:")
try:
    # 获取一个批次来验证标签格式
    sample_batch = next(iter(train_loader))
    sample_data, sample_labels = sample_batch
    print(f"  DataLoader输出标签形状: {sample_labels.shape}")
    print(f"  DataLoader输出标签范围: {sample_labels.min().item()} - {sample_labels.max().item()}")
    print(f"  DataLoader输出标签类型: {sample_labels.dtype}")
    
    # 验证BrainVoxelMatDataset的标签映射是否正确
    if sample_labels.min().item() >= 0 and sample_labels.max().item() <= 101:
        print("  ✅ DataLoader标签格式正确: 0-101 (适用于CrossEntropyLoss)")
    else:
        print(f"  ⚠️ DataLoader标签范围异常: [{sample_labels.min().item()}, {sample_labels.max().item()}]")
        
except Exception as e:
    print(f"  ⚠️ 验证DataLoader标签格式时出错: {e}")

print(f"\n✅ 数据加载完成:")
print(f"  📁 数据格式: MAT格式（固定测试集模式）")
print(f"  📊 训练集样本数: {len(dataset_dict['train_samples'])}")
print(f"  📊 验证集样本数: {len(dataset_dict['val_samples'])}")
print(f"  📊 测试集样本数: {len(dataset_dict['test_samples'])}")
print(f"  🔢 特征维度: {cfg['feature_dim']}")
print(f"  🏷️ 类别数量: {cfg['num_class']}")
print(f"  ⚙️ 数据处理状态:")
print(f"    • 背景像素已在加载阶段过滤 ✓")
print(f"    • StandardScaler标准化已应用 ✓")
print(f"    • 固定prob_idx测试集划分已应用 ✓")
print(f"    • 标签格式已处理（1-102 → 0-101）✓")


In [ ]:
# 单元格6: 初始化模型
# 初始化模型
print(f"🔧 创建模型: {cfg['model_type']}...")

model = get_model(
    model_type=cfg['model_type'],
    input_dim=cfg['feature_dim'],
    hidden_dims=cfg['hidden_units'],
    num_classes=cfg['num_class'],
    dropout_rate=cfg['dropout_rate'],
    activation=cfg['activation']
)
model.to(device)

print(f"✅ 模型创建完成:")
print(f"  模型类型: {cfg['model_type']}")
print(f"  输入维度: {cfg['feature_dim']}")
print(f"  输出类别数: {cfg['num_class']}")
print(f"  隐藏层配置: {cfg['hidden_units']}")
print(f"  激活函数: {cfg['activation']}")
print(f"  Dropout率: {cfg['dropout_rate']}")

# 计算参数量
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  总参数量: {total_params:,}")
print(f"  可训练参数量: {trainable_params:,}")


In [ ]:
# 单元格7: 定义优化器、学习率调度器和损失函数
print("⚖️ 计算类别权重...")

# 🔧 关键修正：确保权重计算使用正确的标签格式
print("🔍 验证权重计算的标签格式...")

from utils.label_processing import create_criterion_with_background_config

# 检查训练标签格式并进行权重计算
train_labels_for_weights = dataset_dict['train_labels']
print(f"权重计算使用的标签形状: {train_labels_for_weights.shape}")

if len(train_labels_for_weights.shape) > 1 and train_labels_for_weights.shape[1] > 1:
    print("权重计算: 检测到one-hot编码，calculate_class_weights会自动处理")
else:
    print("权重计算: 检测到索引格式")

# 🔧 关键修正：使用与main.py完全一致的类别权重计算
class_weights = calculate_class_weights(
    dataset_dict['train_labels'], 
    cfg['num_class'],
    cfg  # 🔧 新增：传递config参数
).to(device)


print(f"类别权重计算完成:")
print(f"  权重数量: {len(class_weights)}")
print(f"  权重范围: [{class_weights.min().item():.4f}, {class_weights.max().item():.4f}]")
print(f"  零权重类别数: {(class_weights == 0).sum().item()}")

# 🔧 关键修正：与main.py完全一致 - 使用类别权重，但不使用ignore_index（因为背景已过滤）
print("📊 创建损失函数: CrossEntropyLoss (使用类别权重，背景已过滤)")
criterion = create_criterion_with_background_config(cfg, class_weights)

ignore_idx = getattr(criterion, 'ignore_index', None)
print(f"📊 损失函数配置:")
print(f"  ignore_index: {ignore_idx if ignore_idx is not None else '无'}")


print(f"🚀 创建优化器: {cfg['optimizer']}")
if cfg['optimizer'].lower() == 'adam':
    optimizer = optim.Adam(
        model.parameters(), 
        lr=cfg['lr'], 
        weight_decay=cfg['weight_decay']
    )
elif cfg['optimizer'].lower() == 'adamw':
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=cfg['lr'], 
        weight_decay=cfg['weight_decay']
    )
else:
    raise ValueError(f"不支持的优化器: {cfg['optimizer']}")

# 创建学习率调度器 - 与main.py完全一致
lr_scheduler = None
if cfg['use_lr_scheduler']:
    print(f"📈 创建学习率调度器: {cfg['lr_scheduler_type']}")
    
    if cfg['lr_scheduler_type'].lower() == 'cosine':
        lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=cfg['epochs']
        )
    elif cfg['lr_scheduler_type'].lower() == 'multistep':
        lr_scheduler = torch.optim.lr_scheduler.MultiStepLR(
            optimizer, 
            milestones=cfg.get('lr_milestones', [10, 20]), 
            gamma=cfg.get('lr_gamma', 0.1)
        )
    elif cfg['lr_scheduler_type'].lower() == 'step':
        lr_scheduler = torch.optim.lr_scheduler.StepLR(
            optimizer, 
            step_size=cfg.get('lr_step_size', 4), 
            gamma=cfg.get('lr_gamma', 0.1)
        )
    elif cfg['lr_scheduler_type'].lower() == 'plateau':
        lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=cfg.get('lr_gamma', 0.1), 
            patience=5, verbose=True
        )
    else:
        print(f"⚠️ 未知的学习率调度器类型: {cfg['lr_scheduler_type']}. 不使用调度器。")
        lr_scheduler = None
else:
    print("🚫 不使用学习率调度器")

# 🔧 关键修正：标准化参数处理 - 与main.py train_and_evaluate函数完全一致
print("📊 处理标准化参数...")

# MAT格式数据已经通过StandardScaler标准化，从dataset_dict中获取scaler
scaler = dataset_dict.get('scaler', None)
normalization_params = None

if scaler is not None:
    print("✅ 检测到StandardScaler对象（MAT数据已标准化）")
    
    # 🔧 关键修正：从StandardScaler提取参数供predict.py使用 - 与main.py一致
    if hasattr(scaler, 'mean_') and hasattr(scaler, 'scale_'):
        # StandardScaler的scale_ = 1/std，所以std = 1/scale_
        std_values = 1.0 / scaler.scale_
        normalization_params = {
            'mean': scaler.mean_.tolist(),
            'std': std_values.tolist()
        }
        print(f"  均值范围: [{scaler.mean_.min():.4f}, {scaler.mean_.max():.4f}]")
        print(f"  标准差范围: [{std_values.min():.4f}, {std_values.max():.4f}]")
        print("  ✅ 标准化参数已提取（用于预测阶段）")
    else:
        print("  ⚠️ StandardScaler对象不完整")
        
    # 将scaler保存到配置中
    cfg['scaler'] = scaler
else:
    print("⚠️ 未找到StandardScaler对象")
    
    # 如果没有scaler但需要标准化参数，尝试从训练数据计算
    if cfg.get('norm', True):
        print("⚠️ 尝试从训练数据计算标准化参数...")
        train_samples = dataset_dict['train_samples']
        mean = np.mean(train_samples, axis=0)
        std = np.std(train_samples, axis=0)
        std[std == 0] = 1e-10  # 避免除零
        
        normalization_params = {
            'mean': mean.tolist(),
            'std': std.tolist()
        }
        print(f"  从训练数据计算的均值范围: [{mean.min():.4f}, {mean.max():.4f}]")
        print(f"  从训练数据计算的标准差范围: [{std.min():.4f}, {std.max():.4f}]")

# 验证数据已经被标准化
print("\n🔍 验证数据标准化状态:")
train_mean = np.mean(dataset_dict['train_samples'], axis=0)
train_std = np.std(dataset_dict['train_samples'], axis=0)
print(f"  训练数据均值范围: [{train_mean.min():.4f}, {train_mean.max():.4f}]")
print(f"  训练数据标准差范围: [{train_std.min():.4f}, {train_std.max():.4f}]")

if abs(train_mean.mean()) < 0.1 and abs(train_std.mean() - 1.0) < 0.1:
    print("  ✅ 数据已正确标准化（均值≈0，标准差≈1）")
else:
    print("  ⚠️ 数据可能未正确标准化")

# 打印训练配置摘要
print(f"\n📋 训练配置摘要:")
print(f"  优化器: {cfg['optimizer']}")
print(f"  学习率: {cfg['lr']:.2e}")
print(f"  权重衰减: {cfg['weight_decay']:.2e}")
print(f"  学习率调度器: {cfg['lr_scheduler_type'] if lr_scheduler else 'None'}")
if lr_scheduler and cfg['lr_scheduler_type'].lower() == 'step':
    print(f"    Step Size: {cfg.get('lr_step_size', 4)}")
    print(f"    Gamma: {cfg.get('lr_gamma', 0.1)}")
print(f"  损失函数: CrossEntropyLoss (使用类别权重)")
print(f"  批处理大小: {cfg['batch_size']}")
print(f"  训练轮数: {cfg['epochs']}")
print(f"  验证频率: 每 {cfg['val_epochs']} 轮")
print(f"  🔧 关键：背景像素已过滤，DataLoader输出标签0-101，模型输出102个类别")

In [ ]:
# 单元格8: 训练与验证循环（修改版 - 支持可配置背景处理）
print(f"🚀 开始训练模型: {cfg.get('model_name', 'BrainVoxelMLP')}")
print(f"  总轮数: {cfg['epochs']}")
print(f"  验证频率: 每 {cfg['val_epochs']} 轮")
print(f"  数据格式: MAT格式")
print(f"  类别数量: {cfg['num_class']}")
print(f"  📊 同时监控测试集性能（不参与模型选择）")

# 🔧 新增：获取背景处理信息
from utils.label_processing import get_ignore_index, get_label_info_string

ignore_index = get_ignore_index(cfg)
label_info = get_label_info_string(cfg)

print(f"  🎯 标签处理: {label_info}")
if ignore_index is not None:
    print(f"  🚫 忽略标签: {ignore_index}")

# 初始化训练结果记录（修改版 - 包含三个数据集的loss）
training_results = {
    'loss_list': [],                # 训练集loss（每epoch）
    'acc_list': [],
    'f1_macro_list': [],
    'val_epoch_list': [],
    'val_acc_list': [],
    'val_f1_macro_list': [],
    'val_kappa_list': [],
    'val_balanced_acc_list': [],
    'val_loss_list': [],            # 新增：验证集loss
    'test_f1_macro_list': [],       # 测试集F1监控
    'test_acc_list': [],            # 测试集准确率监控
    'test_loss_list': [],           # 新增：测试集loss监控
    'train_loss_eval_list': [],     # 新增：训练集评估loss（验证时计算）
    'lr_list': []
}

best_val_f1 = 0.0
best_model_path = os.path.join(save_dir, f"{cfg['experiment_name']}_best_model.pth")

# 开始训练循环
train_start_time = time.time()

for epoch in range(1, cfg['epochs'] + 1):
    current_lr = optimizer.param_groups[0]['lr']
    training_results['lr_list'].append(current_lr)

    model.train()
    epoch_loss = 0
    train_preds_epoch = []
    train_targets_epoch = []
    train_acc = 0
    valid_count = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{cfg['epochs']} [Train]", leave=False)
    for batch_idx, (data, target) in enumerate(progress_bar):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        preds = torch.argmax(output, dim=1)
        
        # 🔧 修改：考虑ignore_index的准确率计算
        if ignore_index is not None:
            # 只计算非ignore标签的准确率
            valid_mask = (target != ignore_index)
            if valid_mask.sum() > 0:
                train_acc += (preds[valid_mask] == target[valid_mask]).sum().item()
                valid_count += valid_mask.sum().item()
                
                # 收集有效的预测和目标
                train_preds_epoch.extend(preds[valid_mask].cpu().numpy())
                train_targets_epoch.extend(target[valid_mask].cpu().numpy())
        else:
            # 所有样本都参与计算
            train_acc += (preds == target).sum().item()
            valid_count += target.size(0)
            
            # 收集预测和目标用于计算F1等指标
            train_preds_epoch.extend(preds.cpu().numpy())
            train_targets_epoch.extend(target.cpu().numpy())

        if batch_idx % 100 == 0:
            progress_bar.set_postfix(loss=loss.item(), lr=current_lr)

    avg_epoch_loss = epoch_loss / len(train_loader)
    epoch_train_acc = train_acc / valid_count if valid_count > 0 else 0.0
    
    # 计算训练集F1分数
    if len(train_preds_epoch) > 0 and len(train_targets_epoch) > 0:
        epoch_train_f1 = f1_score(train_targets_epoch, train_preds_epoch, average='macro')
    else:
        epoch_train_f1 = 0.0

    training_results['loss_list'].append(avg_epoch_loss)
    training_results['acc_list'].append(epoch_train_acc)
    training_results['f1_macro_list'].append(epoch_train_f1)

    print(f"Epoch {epoch}/{cfg['epochs']} - Loss: {avg_epoch_loss:.4f}, Acc: {epoch_train_acc:.4f}, F1: {epoch_train_f1:.4f}, LR: {current_lr:.2e}")

    # 验证阶段（修改版 - 包含三个数据集的loss计算和背景处理）
    if epoch % cfg['val_epochs'] == 0 or epoch == cfg['epochs']:
        model.eval()
        
        # 🔧 新增：训练集评估loss（在验证时计算，不更新梯度）
        train_eval_loss = 0
        train_eval_samples = 0
        with torch.no_grad():
            progress_bar_train_eval = tqdm(train_loader, desc=f"Epoch {epoch}/{cfg['epochs']} [Train Eval]", leave=False)
            for data, target in progress_bar_train_eval:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = criterion(output, target)
                train_eval_loss += loss.item() * data.size(0)  # 累积loss * batch_size
                train_eval_samples += data.size(0)
        
        avg_train_eval_loss = train_eval_loss / train_eval_samples
        
        # 验证集评估（包含loss计算和背景处理）
        val_preds_epoch = []
        val_targets_epoch = []
        val_loss = 0
        val_samples = 0
        val_acc = 0
        val_valid_count = 0
        
        with torch.no_grad():
            progress_bar_val = tqdm(val_loader, desc=f"Epoch {epoch}/{cfg['epochs']} [Val]", leave=False)
            for data, target in progress_bar_val:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = criterion(output, target)
                val_loss += loss.item() * data.size(0)  # 累积loss * batch_size
                val_samples += data.size(0)
                
                preds = torch.argmax(output, dim=1)
                
                # 🔧 修改：考虑ignore_index的验证计算
                if ignore_index is not None:
                    # 只计算非ignore标签
                    valid_mask = (target != ignore_index)
                    if valid_mask.sum() > 0:
                        val_preds_epoch.extend(preds[valid_mask].cpu().numpy())
                        val_targets_epoch.extend(target[valid_mask].cpu().numpy())
                        val_acc += (preds[valid_mask] == target[valid_mask]).sum().item()
                        val_valid_count += valid_mask.sum().item()
                else:
                    # 所有样本都参与计算
                    val_preds_epoch.extend(preds.cpu().numpy())
                    val_targets_epoch.extend(target.cpu().numpy())
                    val_acc += (preds == target).sum().item()
                    val_valid_count += target.size(0)

        avg_val_loss = val_loss / val_samples
        val_accuracy = val_acc / val_valid_count if val_valid_count > 0 else 0.0
        
        # 计算验证集评估指标
        if len(val_preds_epoch) > 0 and len(val_targets_epoch) > 0:
            val_f1_macro = f1_score(val_targets_epoch, val_preds_epoch, average='macro')
            val_kappa = cohen_kappa_score(val_targets_epoch, val_preds_epoch)
            val_balanced_acc = balanced_accuracy_score(val_targets_epoch, val_preds_epoch)
        else:
            val_f1_macro = 0.0
            val_kappa = 0.0
            val_balanced_acc = 0.0

        # 📊 测试集监控评估（包含loss计算和背景处理，不参与模型选择）
        test_preds_epoch = []
        test_targets_epoch = []
        test_loss = 0
        test_samples = 0
        test_acc = 0
        test_valid_count = 0
        
        with torch.no_grad():
            progress_bar_test = tqdm(test_loader, desc=f"Epoch {epoch}/{cfg['epochs']} [Test Monitor]", leave=False)
            for data, target in progress_bar_test:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = criterion(output, target)
                test_loss += loss.item() * data.size(0)  # 累积loss * batch_size
                test_samples += data.size(0)
                
                preds = torch.argmax(output, dim=1)
                
                # 🔧 修改：考虑ignore_index的测试计算
                if ignore_index is not None:
                    # 只计算非ignore标签
                    valid_mask = (target != ignore_index)
                    if valid_mask.sum() > 0:
                        test_preds_epoch.extend(preds[valid_mask].cpu().numpy())
                        test_targets_epoch.extend(target[valid_mask].cpu().numpy())
                        test_acc += (preds[valid_mask] == target[valid_mask]).sum().item()
                        test_valid_count += valid_mask.sum().item()
                else:
                    # 所有样本都参与计算
                    test_preds_epoch.extend(preds.cpu().numpy())
                    test_targets_epoch.extend(target.cpu().numpy())
                    test_acc += (preds == target).sum().item()
                    test_valid_count += target.size(0)

        avg_test_loss = test_loss / test_samples
        test_accuracy = test_acc / test_valid_count if test_valid_count > 0 else 0.0
        
        # 计算测试集F1分数
        if len(test_preds_epoch) > 0 and len(test_targets_epoch) > 0:
            test_f1_macro = f1_score(test_targets_epoch, test_preds_epoch, average='macro')
        else:
            test_f1_macro = 0.0

        # 记录所有结果
        training_results['val_epoch_list'].append(epoch)
        training_results['val_acc_list'].append(val_accuracy)
        training_results['val_f1_macro_list'].append(val_f1_macro)
        training_results['val_kappa_list'].append(val_kappa)
        training_results['val_balanced_acc_list'].append(val_balanced_acc)
        training_results['val_loss_list'].append(avg_val_loss)          # 新增：验证集loss
        
        # 📊 记录测试集监控结果
        training_results['test_f1_macro_list'].append(test_f1_macro)
        training_results['test_acc_list'].append(test_accuracy)
        training_results['test_loss_list'].append(avg_test_loss)        # 新增：测试集loss
        training_results['train_loss_eval_list'].append(avg_train_eval_loss)  # 新增：训练集评估loss

        # 计算差异指标
        train_val_f1_diff = epoch_train_f1 - val_f1_macro
        val_test_f1_diff = val_f1_macro - test_f1_macro
        train_val_loss_diff = avg_train_eval_loss - avg_val_loss
        val_test_loss_diff = avg_val_loss - avg_test_loss
        
        print(f"  Validation - Acc: {val_accuracy:.4f}, F1: {val_f1_macro:.4f}, Loss: {avg_val_loss:.4f}, Kappa: {val_kappa:.4f}")
        print(f"  📊 Test Monitor - Acc: {test_accuracy:.4f}, F1: {test_f1_macro:.4f}, Loss: {avg_test_loss:.4f} (仅监控)")
        print(f"  📊 Train Eval Loss: {avg_train_eval_loss:.4f} (验证时重新计算)")
        print(f"  🔍 F1差异 - Train-Val: {train_val_f1_diff:+.4f}, Val-Test: {val_test_f1_diff:+.4f}")
        print(f"  🔍 Loss差异 - Train-Val: {train_val_loss_diff:+.4f}, Val-Test: {val_test_loss_diff:+.4f}")
        
        # 🔧 新增：显示有效样本统计（当使用ignore_index时）
        if ignore_index is not None:
            print(f"  📈 有效样本统计:")
            print(f"    训练集有效样本: {valid_count}/{len(train_loader.dataset)} ({valid_count/len(train_loader.dataset)*100:.1f}%)")
            print(f"    验证集有效样本: {val_valid_count}/{len(val_loader.dataset)} ({val_valid_count/len(val_loader.dataset)*100:.1f}%)")
            print(f"    测试集有效样本: {test_valid_count}/{len(test_loader.dataset)} ({test_valid_count/len(test_loader.dataset)*100:.1f}%)")

        # 模型保存逻辑（仅基于验证集）
        if val_f1_macro > best_val_f1:
            best_val_f1 = val_f1_macro
            training_info = {
                'epoch': epoch,
                'loss_list': training_results['loss_list'],
                'acc_list': training_results['acc_list'],
                'f1_macro_list': training_results['f1_macro_list'],
                'val_acc_list': training_results['val_acc_list'],
                'val_epoch_list': training_results['val_epoch_list'],
                'val_f1_macro_list': training_results['val_f1_macro_list'],
                'val_kappa_list': training_results['val_kappa_list'],
                'val_balanced_acc_list': training_results['val_balanced_acc_list'],
                'val_loss_list': training_results['val_loss_list'],                      # 新增
                'test_f1_macro_list': training_results['test_f1_macro_list'],
                'test_acc_list': training_results['test_acc_list'],
                'test_loss_list': training_results['test_loss_list'],                    # 新增
                'train_loss_eval_list': training_results['train_loss_eval_list'],        # 新增
                'lr_list': training_results['lr_list'],
                'last_epoch': epoch,
                'train_time': time.time() - train_start_time,
                'num_classes': cfg.get('num_class'),
                'background_processed': label_info,  # 🔧 新增：背景处理信息
                'ignore_index': ignore_index         # 🔧 新增：ignore信息
            }
            
            try:
                _, saved_path = save_model_with_architecture(
                    model=model,
                    optimizer=optimizer,
                    config=cfg,
                    training_info=training_info,
                    normalization_params=normalization_params,
                    save_path=best_model_path,
                    lr_scheduler=lr_scheduler,
                    use_old_zipfile_serialization=cfg.get('use_old_zipfile_serialization', True)
                )
                print(f"   ✅ 最佳模型已保存: {os.path.basename(saved_path)} (Val F1: {best_val_f1:.4f})")
            except Exception as e:
                print(f"   ❌ 模型保存失败: {e}")
                # 备用保存方法
                torch.save({
                    'epoch': epoch,
                    'state_dict': model.state_dict(), 
                    'optimizer': optimizer.state_dict(),
                    'val_f1_macro': val_f1_macro,
                    'config': cfg,
                    'training_info': training_info,
                    'normalization_params': normalization_params,
                    'created_with': f'PyTorch {torch.__version__}',
                    'save_format_version': 1.0,
                    'numpy_version': f'{np.__version__}',
                    'background_info': {  # 🔧 新增：背景处理信息
                        'label_processing': label_info,
                        'ignore_index': ignore_index
                    }
                }, best_model_path, _use_new_zipfile_serialization=not cfg.get('use_old_zipfile_serialization', True))
                print(f"   ✅ 使用备用方法保存模型: {os.path.basename(best_model_path)}")

    # 学习率调度器更新逻辑
    if lr_scheduler is not None:
        if isinstance(lr_scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
            if epoch % cfg['val_epochs'] == 0 or epoch == cfg['epochs']:
                lr_scheduler.step(val_f1_macro)
        else:
            lr_scheduler.step()

total_train_time = time.time() - train_start_time
print(f"\n🎉 训练完成! 总用时: {total_train_time:.2f} 秒")
print(f"📊 最佳验证F1分数: {best_val_f1:.4f}")
print(f"📊 对应的测试F1分数: {training_results['test_f1_macro_list'][training_results['val_f1_macro_list'].index(best_val_f1)]:.4f}")
print(f"🎯 标签处理模式: {label_info}")
if ignore_index is not None:
    print(f"🚫 训练中忽略的标签: {ignore_index}")

In [ ]:
# 单元格9: 可视化训练曲线（增强版 - 包含测试集监控）- 修复版
print("📈 生成训练曲线（包含测试集监控）...")
curves_save_path = os.path.join(save_dir, f"{cfg['experiment_name']}_training_curves_with_test.png")

def plot_enhanced_training_curves(training_results, save_path):
    """
    绘制增强版训练曲线，包含验证集与测试集F1分数对比和三个数据集的loss
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # 1. 训练损失曲线（训练时的loss）
    axes[0, 0].plot(range(1, len(training_results['loss_list']) + 1), 
                    training_results['loss_list'], 'b-', linewidth=2, label='Training Loss (During Training)')
    axes[0, 0].set_title('Training Loss Over Epochs', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Epoch', fontsize=12)
    axes[0, 0].set_ylabel('Loss', fontsize=12)
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend(fontsize=10)
    
    # 2. 三个数据集的评估loss对比
    epochs = training_results['val_epoch_list']
    axes[0, 1].plot(epochs, training_results['train_loss_eval_list'], 'b-', linewidth=2, 
                    marker='o', markersize=4, label='Train Loss (Evaluation)')
    axes[0, 1].plot(epochs, training_results['val_loss_list'], 'g-', linewidth=2, 
                    marker='s', markersize=4, label='Validation Loss')
    axes[0, 1].plot(epochs, training_results['test_loss_list'], 'r--', linewidth=2, 
                    marker='^', markersize=4, label='Test Loss (Monitor)')
    axes[0, 1].set_title('Loss Comparison: Train vs Val vs Test', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Epoch', fontsize=12)
    axes[0, 1].set_ylabel('Loss', fontsize=12)
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend(fontsize=10)
    
    # 3. 准确率对比（训练、验证、测试）
    axes[0, 2].plot(range(1, len(training_results['acc_list']) + 1), 
                    training_results['acc_list'], 'b-', linewidth=2, label='Training Accuracy')
    axes[0, 2].plot(training_results['val_epoch_list'], 
                    training_results['val_acc_list'], 'g-', linewidth=2, marker='o', 
                    markersize=4, label='Validation Accuracy')
    axes[0, 2].plot(training_results['val_epoch_list'], 
                    training_results['test_acc_list'], 'r--', linewidth=2, marker='s', 
                    markersize=4, label='Test Accuracy (Monitor)')
    axes[0, 2].set_title('Accuracy Comparison', fontsize=14, fontweight='bold')
    axes[0, 2].set_xlabel('Epoch', fontsize=12)
    axes[0, 2].set_ylabel('Accuracy', fontsize=12)
    axes[0, 2].grid(True, alpha=0.3)
    axes[0, 2].legend(fontsize=10)
    
    # 4. F1 Macro对比（验证集 vs 测试集） - 重点图表
    axes[1, 0].plot(training_results['val_epoch_list'], 
                    training_results['val_f1_macro_list'], 'g-', linewidth=3, marker='o', 
                    markersize=6, label='Validation F1 Macro', markerfacecolor='white', markeredgewidth=2)
    axes[1, 0].plot(training_results['val_epoch_list'], 
                    training_results['test_f1_macro_list'], 'r--', linewidth=3, marker='s', 
                    markersize=6, label='Test F1 Macro (Monitor)', markerfacecolor='white', markeredgewidth=2)
    
    # 标注最佳验证F1对应的测试F1
    best_val_idx = np.argmax(training_results['val_f1_macro_list'])
    best_val_epoch = training_results['val_epoch_list'][best_val_idx]
    best_val_f1 = training_results['val_f1_macro_list'][best_val_idx]
    best_test_f1 = training_results['test_f1_macro_list'][best_val_idx]
    
    axes[1, 0].scatter([best_val_epoch], [best_val_f1], color='green', s=100, 
                       marker='*', zorder=5, label=f'Best Val F1: {best_val_f1:.4f}')
    axes[1, 0].scatter([best_val_epoch], [best_test_f1], color='red', s=100, 
                       marker='*', zorder=5, label=f'Corresponding Test F1: {best_test_f1:.4f}')
    
    axes[1, 0].set_title('F1 Macro Score: Validation vs Test (Monitor)', fontsize=14, fontweight='bold')
    axes[1, 0].set_xlabel('Epoch', fontsize=12)
    axes[1, 0].set_ylabel('F1 Macro Score', fontsize=12)
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend(fontsize=10, loc='best')
    
    # 添加差异分析文本
    f1_diff = best_val_f1 - best_test_f1
    diff_text = f"Val-Test F1 Gap: {f1_diff:+.4f}"
    axes[1, 0].text(0.05, 0.95, diff_text, transform=axes[1, 0].transAxes, 
                    fontsize=11, verticalalignment='top', 
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    # 5. Loss差异分析
    train_val_loss_diff = [t - v for t, v in zip(training_results['train_loss_eval_list'], 
                                                 training_results['val_loss_list'])]
    val_test_loss_diff = [v - t for v, t in zip(training_results['val_loss_list'], 
                                               training_results['test_loss_list'])]
    
    axes[1, 1].plot(epochs, train_val_loss_diff, 'blue', linewidth=2, marker='o', 
                    markersize=4, label='Train - Val Loss')
    axes[1, 1].plot(epochs, val_test_loss_diff, 'purple', linewidth=2, marker='s', 
                    markersize=4, label='Val - Test Loss')
    axes[1, 1].axhline(y=0, color='black', linestyle='-', alpha=0.5)
    axes[1, 1].set_title('Loss Difference Analysis', fontsize=14, fontweight='bold')
    axes[1, 1].set_xlabel('Epoch', fontsize=12)
    axes[1, 1].set_ylabel('Loss Difference', fontsize=12)
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].legend(fontsize=10)
    
    # 6. 其他验证指标
    axes[1, 2].plot(training_results['val_epoch_list'], 
                    training_results['val_kappa_list'], 'purple', linewidth=2, marker='d', 
                    markersize=4, label='Cohen\'s Kappa')
    axes[1, 2].plot(training_results['val_epoch_list'], 
                    training_results['val_balanced_acc_list'], 'orange', linewidth=2, marker='^', 
                    markersize=4, label='Balanced Accuracy')
    axes[1, 2].set_title('Additional Validation Metrics', fontsize=14, fontweight='bold')
    axes[1, 2].set_xlabel('Epoch', fontsize=12)
    axes[1, 2].set_ylabel('Score', fontsize=12)
    axes[1, 2].grid(True, alpha=0.3)
    axes[1, 2].legend(fontsize=10)
    
    plt.tight_layout(pad=3.0)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    # 打印关键统计信息（包含loss信息）
    print(f"\n📊 关键统计信息:")
    print(f"  最佳验证F1: {best_val_f1:.4f} (Epoch {best_val_epoch})")
    print(f"  对应测试F1: {best_test_f1:.4f}")
    print(f"  验证-测试F1差异: {f1_diff:+.4f}")
    
    # Loss统计信息
    best_train_eval_loss = training_results['train_loss_eval_list'][best_val_idx]
    best_val_loss = training_results['val_loss_list'][best_val_idx]
    best_test_loss = training_results['test_loss_list'][best_val_idx]
    
    print(f"\n📊 最佳模型时的Loss信息:")
    print(f"  训练集评估Loss: {best_train_eval_loss:.4f}")
    print(f"  验证集Loss: {best_val_loss:.4f}")
    print(f"  测试集Loss: {best_test_loss:.4f}")
    print(f"  训练-验证Loss差异: {best_train_eval_loss - best_val_loss:+.4f}")
    print(f"  验证-测试Loss差异: {best_val_loss - best_test_loss:+.4f}")
    
    if len(training_results['val_f1_macro_list']) > 1:
        val_test_correlation = np.corrcoef(training_results['val_f1_macro_list'], 
                                         training_results['test_f1_macro_list'])[0, 1]
        print(f"  验证-测试F1相关性: {val_test_correlation:.4f}")
        
        # Loss相关性
        val_test_loss_correlation = np.corrcoef(training_results['val_loss_list'], 
                                               training_results['test_loss_list'])[0, 1]
        print(f"  验证-测试Loss相关性: {val_test_loss_correlation:.4f}")
        
        # 计算趋势一致性
        val_trend = np.diff(training_results['val_f1_macro_list'])
        test_trend = np.diff(training_results['test_f1_macro_list'])
        trend_consistency = np.mean(np.sign(val_trend) == np.sign(test_trend))
        print(f"  趋势一致性: {trend_consistency:.4f} ({trend_consistency*100:.1f}%)")

try:
    plot_enhanced_training_curves(training_results, curves_save_path)
    print(f"✅ 增强训练曲线已保存: {curves_save_path}")
except Exception as e:
    print(f"❌ 生成增强训练曲线时出错: {e}")
    
    # 备用绘图方案 - 专注于F1对比
    plt.figure(figsize=(12, 8))
    
    plt.subplot(2, 2, 1)
    plt.plot(training_results['loss_list'], 'b-', linewidth=2)
    plt.title('Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 2, 2)
    plt.plot(training_results['acc_list'], 'b-', linewidth=2, label='Training')
    plt.plot(training_results['val_epoch_list'], training_results['val_acc_list'], 
             'g-', linewidth=2, marker='o', label='Validation')
    plt.plot(training_results['val_epoch_list'], training_results['test_acc_list'], 
             'r--', linewidth=2, marker='s', label='Test (Monitor)')
    plt.title('Accuracy Comparison')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 1, 2)
    plt.plot(training_results['val_epoch_list'], training_results['val_f1_macro_list'], 
             'g-', linewidth=3, marker='o', markersize=6, label='Validation F1 Macro')
    plt.plot(training_results['val_epoch_list'], training_results['test_f1_macro_list'], 
             'r--', linewidth=3, marker='s', markersize=6, label='Test F1 Macro (Monitor)')
    
    # 标注最佳点
    best_val_idx = np.argmax(training_results['val_f1_macro_list'])
    best_val_epoch = training_results['val_epoch_list'][best_val_idx]
    best_val_f1 = training_results['val_f1_macro_list'][best_val_idx]
    best_test_f1 = training_results['test_f1_macro_list'][best_val_idx]
    
    plt.scatter([best_val_epoch], [best_val_f1], color='green', s=100, marker='*', zorder=5)
    plt.scatter([best_val_epoch], [best_test_f1], color='red', s=100, marker='*', zorder=5)
    
    plt.title('F1 Macro Score: Validation vs Test Monitoring', fontsize=14, fontweight='bold')
    plt.xlabel('Epoch')
    plt.ylabel('F1 Macro Score')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(curves_save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✅ 使用备用方法生成训练曲线: {curves_save_path}")

In [ ]:
# 单元格10: 在测试集上评估模型（使用项目的评估函数）
print("🧪 开始测试集评估...")

# 加载最佳模型进行测试
if os.path.exists(best_model_path):
    print(f"📂 加载最佳模型: {best_model_path}")
    try:
        # 使用项目的安全加载函数
        from utils.model_io import safe_load_model
        checkpoint = safe_load_model(best_model_path, device)
        
        # 检查检查点格式并加载权重
        if 'state_dict' in checkpoint:
            model.load_state_dict(checkpoint['state_dict'])
        elif 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
        else:
            raise KeyError("检查点中未找到模型权重")
            
        print("✅ 成功加载最佳模型权重")
    except Exception as e:
        print(f"⚠️ 加载最佳模型失败: {e}，使用当前模型状态")
else:
    print("⚠️ 未找到最佳模型检查点，使用当前模型状态进行测试")

# 使用项目的完整评估函数 - 与main.py调用方式完全一致
print("\n📊 在测试集上进行完整评估...")
test_results = evaluate_model(
    model=model,
    data_loader=test_loader,
    device=device,
    result_path=save_dir,
    dataset_name="test",
    detailed=True,
    plot=True,
    disable_progress=False,
    show_class_metrics=True,
    config=cfg  # 🔧 新增：传递config参数
)

print("\n🎯 测试集最终结果:")
print(f"  准确率: {test_results['accuracy']:.4f}")
print(f"  宏平均F1: {test_results['f1_macro']:.4f}")
print(f"  加权F1: {test_results['f1_weighted']:.4f}")
print(f"  平衡准确率: {test_results['balanced_accuracy']:.4f}")
print(f"  Cohen's Kappa: {test_results['kappa']:.4f}")
print(f"  预测类别数: {len(test_results['unique_classes'])}")

# 额外的完整数据集评估（与main.py一致）
print("\n📈 进行完整数据集评估对比...")

# 在训练集上评估
print("\n在训练集上评估...")
train_results = evaluate_model(
    model=model,
    data_loader=train_loader,
    device=device,
    result_path=save_dir,
    dataset_name="train",
    detailed=True,
    plot=True,
    disable_progress=True,
    show_class_metrics=True
)

# 在验证集上评估
print("\n在验证集上评估...")
val_results = evaluate_model(
    model=model,
    data_loader=val_loader,
    device=device,
    result_path=save_dir,
    dataset_name="val",
    detailed=True,
    plot=True,
    disable_progress=True,
    show_class_metrics=True
)

print("\n📊 完整评估结果对比:")
print(f"训练集 - 准确率: {train_results['accuracy']:.4f}, F1: {train_results['f1_macro']:.4f}, Kappa: {train_results['kappa']:.4f}")
print(f"验证集 - 准确率: {val_results['accuracy']:.4f}, F1: {val_results['f1_macro']:.4f}, Kappa: {val_results['kappa']:.4f}")
print(f"测试集 - 准确率: {test_results['accuracy']:.4f}, F1: {test_results['f1_macro']:.4f}, Kappa: {test_results['kappa']:.4f}")

# 使用项目的类别性能比较函数
try:
    from utils.metrics import compare_class_performance
    compare_results = compare_class_performance(
        results_list=[train_results, val_results, test_results],
        dataset_names=["Train", "Validation", "Test"],
        result_path=save_dir
    )
    print("✅ 类别性能比较完成，结果已保存")
except Exception as e:
    print(f"⚠️ 类别性能比较时出错: {e}")

In [ ]:
# 单元格11: 额外的可视化和分析
print("📊 生成额外的可视化结果...")

# 混淆矩阵可视化
try:
    cm_save_path = os.path.join(save_dir, f"{cfg['experiment_name']}_confusion_matrix.png")
    visualize_confusion_matrix(
        test_results['confusion_matrix'], 
        save_path=cm_save_path, 
        log_scale=True
    )
    print(f"✅ 混淆矩阵已保存: {cm_save_path}")
except Exception as e:
    print(f"⚠️ 生成混淆矩阵时出错: {e}")

# 生成类别性能分析
if len(test_results['unique_classes']) > 0:
    print("\n📈 类别性能分析:")
    class_f1_scores = test_results['class_f1']
    unique_classes = test_results['unique_classes']
    
    # 找出表现最好和最差的类别
    best_class_idx = np.argmax(class_f1_scores) if len(class_f1_scores) > 0 else 0
    worst_class_idx = np.argmin(class_f1_scores) if len(class_f1_scores) > 0 else 0
    
    if len(class_f1_scores) > 0:
        print(f"  表现最好的类别: {unique_classes[best_class_idx]} (F1: {class_f1_scores[best_class_idx]:.4f})")
        print(f"  表现最差的类别: {unique_classes[worst_class_idx]} (F1: {class_f1_scores[worst_class_idx]:.4f})")
        print(f"  平均F1分数: {np.mean(class_f1_scores):.4f}")
        print(f"  F1分数标准差: {np.std(class_f1_scores):.4f}")

# 绘制F1分数分布
if len(test_results['class_f1']) > 5:  # 只有足够多的类别时才绘制
    plt.figure(figsize=(12, 6))
    
    plt.subplot(1, 2, 1)
    plt.hist(test_results['class_f1'], bins=20, alpha=0.7, color='skyblue', edgecolor='black')
    plt.xlabel('F1 Score')
    plt.ylabel('Number of Classes')
    plt.title('Distribution of F1 Scores Across Classes')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    sorted_f1 = np.sort(test_results['class_f1'])
    plt.plot(sorted_f1, 'bo-', markersize=3)
    plt.xlabel('Class Rank (sorted by F1)')
    plt.ylabel('F1 Score')
    plt.title('F1 Scores Sorted by Performance')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    f1_dist_path = os.path.join(save_dir, f"{cfg['experiment_name']}_f1_distribution.png")
    plt.savefig(f1_dist_path)
    plt.show()
    print(f"✅ F1分数分布图已保存: {f1_dist_path}")


In [ ]:
# 单元格11: 详细的测试集监控分析（包含loss分析）
print("🔍 开始详细的测试集监控分析...")

# 补充代码：详细的测试集监控分析（包含loss分析）
def analyze_test_monitoring_results(training_results):
    """
    分析验证集与测试集的监控结果，包含loss和F1的详细统计分析
    """
    print("\n" + "="*60)
    print("🔍 测试集监控结果详细分析（包含Loss）")
    print("="*60)
    
    val_f1_list = training_results['val_f1_macro_list']
    test_f1_list = training_results['test_f1_macro_list']
    val_loss_list = training_results['val_loss_list']
    test_loss_list = training_results['test_loss_list']
    train_loss_eval_list = training_results['train_loss_eval_list']
    epoch_list = training_results['val_epoch_list']
    
    # 1. 基本统计（F1和Loss）
    print(f"\n📊 基本统计信息:")
    print(f"  验证集F1 - 均值: {np.mean(val_f1_list):.4f}, 标准差: {np.std(val_f1_list):.4f}")
    print(f"  测试集F1 - 均值: {np.mean(test_f1_list):.4f}, 标准差: {np.std(test_f1_list):.4f}")
    print(f"  验证-测试F1差异 - 均值: {np.mean(np.array(val_f1_list) - np.array(test_f1_list)):.4f}")
    
    print(f"\n📊 Loss统计信息:")
    print(f"  训练集Loss(评估) - 均值: {np.mean(train_loss_eval_list):.4f}, 标准差: {np.std(train_loss_eval_list):.4f}")
    print(f"  验证集Loss - 均值: {np.mean(val_loss_list):.4f}, 标准差: {np.std(val_loss_list):.4f}")
    print(f"  测试集Loss - 均值: {np.mean(test_loss_list):.4f}, 标准差: {np.std(test_loss_list):.4f}")
    
    # 2. 最佳模型对应的测试性能
    best_val_idx = np.argmax(val_f1_list)
    best_val_epoch = epoch_list[best_val_idx]
    best_val_f1 = val_f1_list[best_val_idx]
    corresponding_test_f1 = test_f1_list[best_val_idx]
    corresponding_train_loss = train_loss_eval_list[best_val_idx]
    corresponding_val_loss = val_loss_list[best_val_idx]
    corresponding_test_loss = test_loss_list[best_val_idx]
    
    print(f"\n🏆 最佳模型性能:")
    print(f"  最佳验证F1: {best_val_f1:.4f} (Epoch {best_val_epoch})")
    print(f"  对应测试F1: {corresponding_test_f1:.4f}")
    print(f"  验证-测试F1差异: {best_val_f1 - corresponding_test_f1:+.4f}")
    print(f"  对应训练Loss: {corresponding_train_loss:.4f}")
    print(f"  对应验证Loss: {corresponding_val_loss:.4f}")
    print(f"  对应测试Loss: {corresponding_test_loss:.4f}")
    print(f"  训练-验证Loss差异: {corresponding_train_loss - corresponding_val_loss:+.4f}")
    print(f"  验证-测试Loss差异: {corresponding_val_loss - corresponding_test_loss:+.4f}")
    
    # 3. 测试集上的最佳性能
    best_test_idx = np.argmax(test_f1_list)
    best_test_epoch = epoch_list[best_test_idx]
    best_test_f1 = test_f1_list[best_test_idx]
    corresponding_val_f1 = val_f1_list[best_test_idx]
    
    print(f"\n🎯 测试集最佳性能:")
    print(f"  测试集最佳F1: {best_test_f1:.4f} (Epoch {best_test_epoch})")
    print(f"  对应验证F1: {corresponding_val_f1:.4f}")
    print(f"  如果在测试集上选模型的话，损失: {best_test_f1 - corresponding_test_f1:+.4f}")
    
    # 4. 相关性分析（F1和Loss）
    if len(val_f1_list) > 2:
        f1_correlation = np.corrcoef(val_f1_list, test_f1_list)[0, 1]
        loss_correlation = np.corrcoef(val_loss_list, test_loss_list)[0, 1]
        
        print(f"\n📈 相关性分析:")
        print(f"  验证-测试F1相关系数: {f1_correlation:.4f}")
        print(f"  验证-测试Loss相关系数: {loss_correlation:.4f}")
        
        if f1_correlation > 0.8:
            print("  ✅ F1高度正相关 - 验证集是测试集的良好代理")
        elif f1_correlation > 0.6:
            print("  ✅ F1中等正相关 - 验证集基本可以代表测试集趋势")
        elif f1_correlation > 0.3:
            print("  ⚠️ F1弱正相关 - 验证集与测试集存在一定差异")
        else:
            print("  ❌ F1相关性较低 - 需要谨慎解释结果")
            
        if loss_correlation > 0.8:
            print("  ✅ Loss高度正相关 - 验证Loss是测试Loss的良好指标")
        elif loss_correlation > 0.6:
            print("  ✅ Loss中等正相关 - 验证Loss基本反映测试Loss趋势")
        else:
            print("  ⚠️ Loss相关性较低 - 验证Loss与测试Loss存在差异")
    
    # 5. 趋势一致性分析
    if len(val_f1_list) > 2:
        val_f1_diff = np.diff(val_f1_list)
        test_f1_diff = np.diff(test_f1_list)
        f1_trend_agreement = np.mean(np.sign(val_f1_diff) == np.sign(test_f1_diff))
        
        val_loss_diff = np.diff(val_loss_list)
        test_loss_diff = np.diff(test_loss_list)
        loss_trend_agreement = np.mean(np.sign(val_loss_diff) == np.sign(test_loss_diff))
        
        print(f"\n📊 趋势一致性:")
        print(f"  F1趋势一致性: {f1_trend_agreement:.3f} ({f1_trend_agreement*100:.1f}%)")
        print(f"  Loss趋势一致性: {loss_trend_agreement:.3f} ({loss_trend_agreement*100:.1f}%)")
        
        if f1_trend_agreement > 0.8:
            print("  ✅ 验证集与测试集F1趋势高度一致")
        elif f1_trend_agreement > 0.6:
            print("  ✅ 验证集与测试集F1趋势基本一致")
        else:
            print("  ⚠️ 验证集与测试集F1趋势存在分歧")
    
    # 6. 过拟合风险评估（基于F1和Loss）
    final_val_f1 = val_f1_list[-1]
    final_test_f1 = test_f1_list[-1]
    final_f1_gap = final_val_f1 - final_test_f1
    
    final_train_loss = train_loss_eval_list[-1]
    final_val_loss = val_loss_list[-1]
    final_test_loss = test_loss_list[-1]
    final_train_val_loss_gap = final_train_loss - final_val_loss
    final_val_test_loss_gap = final_val_loss - final_test_loss
    
    print(f"\n⚖️ 过拟合风险评估:")
    print(f"  最终验证F1: {final_val_f1:.4f}")
    print(f"  最终测试F1: {final_test_f1:.4f}")
    print(f"  最终F1性能差距: {final_f1_gap:+.4f}")
    
    print(f"  最终训练Loss: {final_train_loss:.4f}")
    print(f"  最终验证Loss: {final_val_loss:.4f}")
    print(f"  最终测试Loss: {final_test_loss:.4f}")
    print(f"  最终训练-验证Loss差距: {final_train_val_loss_gap:+.4f}")
    print(f"  最终验证-测试Loss差距: {final_val_test_loss_gap:+.4f}")
    
    # 过拟合风险判断
    overfitting_risk = False
    if final_f1_gap > 0.05:
        print("  ⚠️ 可能存在过拟合风险 (验证F1明显优于测试F1)")
        overfitting_risk = True
    elif final_f1_gap < -0.05:
        print("  📈 测试F1优于验证F1 (可能验证集较为严格)")
    else:
        print("  ✅ 验证集与测试集F1接近，泛化良好")
    
    if final_train_val_loss_gap < -0.1:  # 训练loss明显小于验证loss
        print("  ⚠️ 训练Loss明显低于验证Loss，可能存在过拟合")
        overfitting_risk = True
    elif abs(final_train_val_loss_gap) < 0.05:
        print("  ✅ 训练Loss与验证Loss接近，模型拟合良好")
    
    return {
        'best_val_f1': best_val_f1,
        'corresponding_test_f1': corresponding_test_f1,
        'best_test_f1': best_test_f1,
        'f1_correlation': f1_correlation if len(val_f1_list) > 2 else None,
        'loss_correlation': loss_correlation if len(val_f1_list) > 2 else None,
        'f1_trend_agreement': f1_trend_agreement if len(val_f1_list) > 2 else None,
        'loss_trend_agreement': loss_trend_agreement if len(val_f1_list) > 2 else None,
        'final_f1_gap': final_f1_gap,
        'final_train_val_loss_gap': final_train_val_loss_gap,
        'final_val_test_loss_gap': final_val_test_loss_gap,
        'overfitting_risk': overfitting_risk
    }

# 执行详细监控分析
monitoring_analysis = analyze_test_monitoring_results(training_results)

# 创建专门的F1和Loss对比图
def plot_f1_loss_comparison_detail(training_results, save_path):
    """创建详细的F1和Loss对比图"""
    plt.figure(figsize=(16, 12))
    
    epochs = training_results['val_epoch_list']
    val_f1 = training_results['val_f1_macro_list']
    test_f1 = training_results['test_f1_macro_list']
    train_loss_eval = training_results['train_loss_eval_list']
    val_loss = training_results['val_loss_list']
    test_loss = training_results['test_loss_list']
    
    # 主图1：F1对比 (上半部分)
    plt.subplot(3, 2, (1, 2))  # 占据上方两个位置
    
    # 绘制F1曲线
    plt.plot(epochs, val_f1, 'g-', linewidth=3, marker='o', markersize=6, 
             markerfacecolor='white', markeredgewidth=2, label='Validation F1 Macro')
    plt.plot(epochs, test_f1, 'r--', linewidth=3, marker='s', markersize=6, 
             markerfacecolor='white', markeredgewidth=2, label='Test F1 Macro (Monitor Only)')
    
    # 标注最佳点
    best_val_idx = np.argmax(val_f1)
    best_val_epoch = epochs[best_val_idx]
    best_val_f1_score = val_f1[best_val_idx]
    best_test_f1_score = test_f1[best_val_idx]
    
    plt.scatter([best_val_epoch], [best_val_f1_score], color='green', s=150, 
                marker='*', zorder=5, edgecolors='darkgreen', linewidth=2)
    plt.scatter([best_val_epoch], [best_test_f1_score], color='red', s=150, 
                marker='*', zorder=5, edgecolors='darkred', linewidth=2)
    
    plt.axvline(x=best_val_epoch, color='gray', linestyle=':', alpha=0.7, 
                label=f'Best Model Selection (Epoch {best_val_epoch})')
    
    plt.title('F1 Macro Score Comparison: Validation vs Test Monitoring', 
              fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Epoch', fontsize=13)
    plt.ylabel('F1 Macro Score', fontsize=13)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=11, loc='best')
    
    # 主图2：Loss对比 (中间部分)
    plt.subplot(3, 2, (3, 4))  # 占据中间两个位置
    
    plt.plot(epochs, train_loss_eval, 'b-', linewidth=3, marker='o', markersize=6, 
             markerfacecolor='white', markeredgewidth=2, label='Train Loss (Evaluation)')
    plt.plot(epochs, val_loss, 'g-', linewidth=3, marker='s', markersize=6, 
             markerfacecolor='white', markeredgewidth=2, label='Validation Loss')
    plt.plot(epochs, test_loss, 'r--', linewidth=3, marker='^', markersize=6, 
             markerfacecolor='white', markeredgewidth=2, label='Test Loss (Monitor Only)')
    
    # 标注最佳验证F1时的loss值
    best_train_loss = train_loss_eval[best_val_idx]
    best_val_loss = val_loss[best_val_idx]
    best_test_loss = test_loss[best_val_idx]
    
    plt.scatter([best_val_epoch], [best_train_loss], color='blue', s=100, 
                marker='*', zorder=5, edgecolors='darkblue', linewidth=2)
    plt.scatter([best_val_epoch], [best_val_loss], color='green', s=100, 
                marker='*', zorder=5, edgecolors='darkgreen', linewidth=2)
    plt.scatter([best_val_epoch], [best_test_loss], color='red', s=100, 
                marker='*', zorder=5, edgecolors='darkred', linewidth=2)
    
    plt.axvline(x=best_val_epoch, color='gray', linestyle=':', alpha=0.7)
    
    plt.title('Loss Comparison: Train vs Validation vs Test', 
              fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Epoch', fontsize=13)
    plt.ylabel('Loss', fontsize=13)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=11, loc='best')
    
    # 左下：F1差异分析
    plt.subplot(3, 2, 5)
    f1_gaps = np.array(val_f1) - np.array(test_f1)
    plt.plot(epochs, f1_gaps, 'purple', linewidth=2, marker='d', markersize=4)
    plt.axhline(y=0, color='black', linestyle='-', alpha=0.5)
    plt.axhline(y=0.05, color='red', linestyle='--', alpha=0.5, label='Potential Overfitting')
    plt.axhline(y=-0.05, color='blue', linestyle='--', alpha=0.5, label='Test > Validation')
    
    plt.title('F1 Gap Analysis (Val - Test)', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch', fontsize=11)
    plt.ylabel('F1 Gap', fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=9)
    
    # 右下：Loss差异分析
    plt.subplot(3, 2, 6)
    train_val_loss_gaps = np.array(train_loss_eval) - np.array(val_loss)
    val_test_loss_gaps = np.array(val_loss) - np.array(test_loss)
    
    plt.plot(epochs, train_val_loss_gaps, 'blue', linewidth=2, marker='o', markersize=4, 
             label='Train - Val Loss')
    plt.plot(epochs, val_test_loss_gaps, 'purple', linewidth=2, marker='s', markersize=4, 
             label='Val - Test Loss')
    plt.axhline(y=0, color='black', linestyle='-', alpha=0.5)
    
    plt.title('Loss Gap Analysis', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch', fontsize=11)
    plt.ylabel('Loss Gap', fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=9)
    
    plt.tight_layout(pad=3.0)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    # 计算统计信息
    f1_gap = best_val_f1_score - best_test_f1_score
    f1_correlation = np.corrcoef(val_f1, test_f1)[0, 1]
    loss_correlation = np.corrcoef(val_loss, test_loss)[0, 1]
    
    return f1_gap, f1_correlation, loss_correlation

# 执行详细F1和Loss可视化
f1_loss_detail_path = os.path.join(save_dir, f"{cfg['experiment_name']}_f1_loss_comparison_detail.png")
try:
    f1_gap_result, f1_corr_result, loss_corr_result = plot_f1_loss_comparison_detail(training_results, f1_loss_detail_path)
    print(f"✅ 详细F1和Loss对比图已保存: {f1_loss_detail_path}")
    print(f"📊 F1性能差距: {f1_gap_result:+.4f}")
    print(f"📊 F1相关系数: {f1_corr_result:.4f}")
    print(f"📊 Loss相关系数: {loss_corr_result:.4f}")
except Exception as e:
    print(f"❌ 生成详细F1和Loss对比图时出错: {e}")

# 保存包含Loss的完整监控结果到CSV文件
def save_complete_monitoring_results_to_csv(training_results, save_dir, experiment_name):
    """将完整的监控结果（包含Loss）保存为CSV文件"""
    import pandas as pd
    
    # 创建完整监控数据DataFrame
    monitoring_data = {
        'epoch': training_results['val_epoch_list'],
        'validation_f1_macro': training_results['val_f1_macro_list'],
        'test_f1_macro_monitor': training_results['test_f1_macro_list'],
        'validation_accuracy': training_results['val_acc_list'],
        'test_accuracy_monitor': training_results['test_acc_list'],
        'validation_kappa': training_results['val_kappa_list'],
        'validation_balanced_acc': training_results['val_balanced_acc_list'],
        'train_loss_evaluation': training_results['train_loss_eval_list'],
        'validation_loss': training_results['val_loss_list'],
        'test_loss_monitor': training_results['test_loss_list'],
        'f1_gap_val_minus_test': [v - t for v, t in zip(training_results['val_f1_macro_list'], 
                                                        training_results['test_f1_macro_list'])],
        'loss_gap_train_minus_val': [t - v for t, v in zip(training_results['train_loss_eval_list'],
                                                           training_results['val_loss_list'])],
        'loss_gap_val_minus_test': [v - t for v, t in zip(training_results['val_loss_list'],
                                                          training_results['test_loss_list'])]
    }
    
    df = pd.DataFrame(monitoring_data)
    csv_path = os.path.join(save_dir, f"{experiment_name}_complete_monitoring_results.csv")
    df.to_csv(csv_path, index=False)
    print(f"✅ 完整监控结果已保存至CSV: {csv_path}")
    
    # 显示关键统计
    print(f"\n📋 完整监控结果摘要:")
    print(f"  验证集F1最大值: {df['validation_f1_macro'].max():.4f}")
    print(f"  测试集F1最大值: {df['test_f1_macro_monitor'].max():.4f}")
    print(f"  平均F1差距: {df['f1_gap_val_minus_test'].mean():.4f}")
    print(f"  F1差距标准差: {df['f1_gap_val_minus_test'].std():.4f}")
    print(f"  验证Loss最小值: {df['validation_loss'].min():.4f}")
    print(f"  测试Loss最小值: {df['test_loss_monitor'].min():.4f}")
    print(f"  平均训练-验证Loss差距: {df['loss_gap_train_minus_val'].mean():.4f}")
    print(f"  平均验证-测试Loss差距: {df['loss_gap_val_minus_test'].mean():.4f}")
    
    return csv_path

# 保存完整监控结果
try:
    complete_csv_path = save_complete_monitoring_results_to_csv(training_results, save_dir, cfg['experiment_name'])
except Exception as e:
    print(f"⚠️ 保存完整CSV文件时出错: {e}")

print("\n" + "="*60)
print("🎯 详细监控分析完成")
print("="*60)
print("✅ 已完成验证集与测试集的F1和Loss详细分析")
print("✅ 已生成专门的F1和Loss对比详细图表")
print("✅ 已计算相关性、趋势一致性和过拟合风险")
print("✅ 已保存包含所有指标的完整CSV文件")
print("✅ 可进行后续的最终结果保存")
print("="*60)

In [ ]:

# 单元格12: 保存最终实验结果（修改后）
print("💾 保存最终实验结果...")

# 🔧 修改：创建与新配置一致的结果结构
final_results = {
    'experiment_info': {
        'experiment_name': cfg['experiment_name'],
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        'data_format': 'MAT格式（固定测试集）',
        'total_train_time': total_train_time,
        'pytorch_version': torch.__version__,
        'numpy_version': np.__version__,
        'use_bayesian_optimized_params': True,
        'data_split_mode': 'fixed_prob_idx' if cfg.get('test_prob_idx') else 'random'
    },
    'data_split_info': {
        'split_mode': 'fixed_prob_idx',
        'test_prob_idx_configured': cfg.get('test_prob_idx', []),
        'test_prob_idx_actual': dataset_dict.get('test_prob_idx_actual', []),
        'train_val_prob_idx': dataset_dict.get('train_val_prob_idx', []),
        'train_val_ratio': cfg.get('train_val_ratio', 0.75),
        'data_cross_validation': 'passed' if not set(dataset_dict.get('test_prob_idx_actual', [])).intersection(set(dataset_dict.get('train_val_prob_idx', []))) else 'failed'
    },
    'hyperparameters_used': {
        'model_type': cfg['model_type'],
        'hidden_units': cfg['hidden_units'],
        'activation': cfg['activation'],
        'dropout_rate': cfg['dropout_rate'],
        'optimizer': cfg['optimizer'],
        'learning_rate': cfg['lr'],
        'weight_decay': cfg['weight_decay'],
        'lr_scheduler_type': cfg['lr_scheduler_type'],
        'lr_step_size': cfg.get('lr_step_size', 4),
        'lr_gamma': cfg.get('lr_gamma', 0.1),
        'batch_size': cfg['batch_size'],
        'epochs': cfg['epochs'],
        'val_epochs': cfg['val_epochs'],
        'random_seed': cfg['random_seed']
    },
    'data_info': {
        'mat_file_path': cfg['mat_file_path'],
        'feature_dim': cfg['feature_dim'],
        'num_classes': cfg['num_class'],
        'train_samples': len(dataset_dict['train_samples']), 
        'val_samples': len(dataset_dict['val_samples']),
        'test_samples': len(dataset_dict['test_samples']),
        'data_processing': '背景像素已在加载阶段过滤，使用固定prob_idx测试集',
        'normalization': 'StandardScaler应用于特征（仅在训练集上拟合）',
        'test_set_configuration': {
            'type': 'fixed_prob_idx',
            'prob_idx_list': dataset_dict.get('test_prob_idx_actual', []),
            'sample_count': len(dataset_dict['test_samples'])
        }
    },
    'training_results': training_results,
    'final_performance': {
        'best_validation_f1': best_val_f1,
        # 训练集性能
        'train_accuracy': train_results['accuracy'],
        'train_f1_macro': train_results['f1_macro'],
        'train_f1_weighted': train_results['f1_weighted'],
        'train_balanced_accuracy': train_results['balanced_accuracy'],
        'train_kappa': train_results['kappa'],
        # 验证集性能
        'val_accuracy': val_results['accuracy'],
        'val_f1_macro': val_results['f1_macro'],
        'val_f1_weighted': val_results['f1_weighted'],
        'val_balanced_accuracy': val_results['balanced_accuracy'],
        'val_kappa': val_results['kappa'],
        # 测试集性能
        'test_accuracy': test_results['accuracy'],
        'test_f1_macro': test_results['f1_macro'],
        'test_f1_weighted': test_results['f1_weighted'],
        'test_balanced_accuracy': test_results['balanced_accuracy'],
        'test_kappa': test_results['kappa'],
        'predicted_classes_count': len(test_results['unique_classes'])
    },
    'bayesian_optimization_info': {
        'original_hyperparams': best_hyperparams_from_bayesian,
        'source': '贝叶斯优化得到的最佳超参数',
        'optimization_metric': 'F1_macro'
    }
}

# 保存为JSON
results_json_path = os.path.join(save_dir, f"{cfg['experiment_name']}_complete_results.json")
with open(results_json_path, 'w', encoding='utf-8') as f:
    json.dump(final_results, f, indent=4, ensure_ascii=False)
print(f"✅ 完整结果已保存: {results_json_path}")

# 保存实验配置
config_save_path = os.path.join(save_dir, f"{cfg['experiment_name']}_config.json")
config_to_save = cfg.copy()
# 移除不可序列化的对象
if 'scaler' in config_to_save:
    del config_to_save['scaler']
if 'pca_model' in config_to_save:
    del config_to_save['pca_model']

save_config(config_to_save, config_save_path)
print(f"✅ 实验配置已保存: {config_save_path}")

# 🔧 修改：创建最终报告文本
final_report_path = os.path.join(save_dir, f"{cfg['experiment_name']}_final_report.txt")
with open(final_report_path, 'w', encoding='utf-8') as f:
    f.write(f"脑体素分类实验最终报告（固定测试集）\n")
    f.write(f"{'='*50}\n\n")
    f.write(f"实验名称: {cfg['experiment_name']}\n")
    f.write(f"完成时间: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"数据格式: MAT格式（固定prob_idx测试集）\n")
    f.write(f"训练时长: {total_train_time:.2f} 秒\n")
    f.write(f"使用贝叶斯优化最佳超参数: 是\n\n")
    
    f.write(f"数据划分配置:\n")
    f.write(f"划分模式: 固定prob_idx测试集\n")
    f.write(f"测试集prob_idx: {dataset_dict.get('test_prob_idx_actual', [])}\n")
    f.write(f"训练/验证比例: {cfg.get('train_val_ratio', 0.75):.2f}/{1-cfg.get('train_val_ratio', 0.75):.2f}\n")
    f.write(f"数据集交叉验证: {'通过' if final_results['data_split_info']['data_cross_validation'] == 'passed' else '失败'}\n\n")
    
    f.write(f"模型架构: {cfg['model_type']}\n")
    f.write(f"隐藏层配置: {cfg['hidden_units']}\n")
    f.write(f"激活函数: {cfg['activation']}\n")
    f.write(f"Dropout率: {cfg['dropout_rate']}\n")
    f.write(f"优化器: {cfg['optimizer']}\n")
    f.write(f"学习率: {cfg['lr']}\n")
    f.write(f"权重衰减: {cfg['weight_decay']}\n\n")
    
    f.write(f"数据信息:\n")
    f.write(f"特征维度: {cfg['feature_dim']}\n")
    f.write(f"类别数量: {cfg['num_class']}\n")
    f.write(f"训练样本: {len(dataset_dict['train_samples'])}\n")
    f.write(f"验证样本: {len(dataset_dict['val_samples'])}\n")
    f.write(f"测试样本: {len(dataset_dict['test_samples'])}\n")
    f.write(f"数据处理: 背景像素已过滤，固定prob_idx测试集\n\n")
    
    f.write(f"训练集性能:\n")
    f.write(f"  准确率: {train_results['accuracy']:.4f}\n")
    f.write(f"  宏平均F1: {train_results['f1_macro']:.4f}\n")
    f.write(f"  平衡准确率: {train_results['balanced_accuracy']:.4f}\n")
    f.write(f"  Kappa系数: {train_results['kappa']:.4f}\n\n")
    
    f.write(f"验证集性能:\n")
    f.write(f"  准确率: {val_results['accuracy']:.4f}\n")
    f.write(f"  宏平均F1: {val_results['f1_macro']:.4f}\n")
    f.write(f"  平衡准确率: {val_results['balanced_accuracy']:.4f}\n")
    f.write(f"  Kappa系数: {val_results['kappa']:.4f}\n\n")
    
    f.write(f"测试集性能:\n")
    f.write(f"  准确率: {test_results['accuracy']:.4f}\n")
    f.write(f"  宏平均F1: {test_results['f1_macro']:.4f}\n")
    f.write(f"  平衡准确率: {test_results['balanced_accuracy']:.4f}\n")
    f.write(f"  Kappa系数: {test_results['kappa']:.4f}\n\n")
    
    f.write(f"测试集配置详情:\n")
    f.write(f"  类型: 固定prob_idx划分\n")
    f.write(f"  prob_idx列表: {dataset_dict.get('test_prob_idx_actual', [])}\n")
    f.write(f"  样本数量: {len(dataset_dict['test_samples'])}\n\n")
    
    f.write(f"最佳模型保存路径: {best_model_path}\n")
    f.write(f"所有结果保存目录: {save_dir}\n")

print(f"✅ 最终报告已保存: {final_report_path}")

# 打印实验总结
print(f"\n🎉 实验完成总结（固定测试集模式）:")
print(f"  📁 结果目录: {save_dir}")
print(f"  🎯 测试集配置: prob_idx {dataset_dict.get('test_prob_idx_actual', [])}")
print(f"  🏆 最佳验证F1: {best_val_f1:.4f}")
print(f"  🎯 测试集F1: {test_results['f1_macro']:.4f}")
print(f"  🎯 测试集准确率: {test_results['accuracy']:.4f}")
print(f"  🎯 测试集Kappa: {test_results['kappa']:.4f}")
print(f"  ⏱️ 总训练时间: {total_train_time:.2f} 秒")
print(f"  📊 数据划分: 固定prob_idx测试集，无数据泄露")
print(f"  📈 所有可视化和详细结果已保存")
print(f"  🏆 最佳模型: {os.path.basename(best_model_path)}")

print("\n✨ Notebook执行完毕!（固定测试集配置）")